## Recommended Starting Hyperparameters
1. Population Size
* Typical Range: 50–500.
* Recommendation: Start with 100–200.
* Larger populations explore more, but are slower per generation.
* With your large search space, erring toward the higher end (e.g., 200) is reasonable if your fitness function is fast.

2. Number of Generations
* Typical Range: 50–500+.
* Recommendation: 100–300 generations.
* More generations allow more refinement, but diminishing returns after a point.
* You can monitor if the best fitness plateaus early.
3. Selection Method
* Tournament selection or roulette wheel selection are both good.
* Many libraries use tournament selection by default.
4. Crossover Rate
* Typical: 0.7–0.9 (i.e., 70–90% of offspring are produced by crossover).
* Recommendation: 0.8 (80%).
5. Mutation Rate
* Typical: 0.01–0.1 (1–10% mutation rate per gene).
* Recommendation: 0.05 (5% per gene).
* With 9 genes (slots), this means about half your children will have at least one mutation per generation.
6. Elitism
* Keep the best 1–2 individuals from each generation to ensure you don’t lose the best found so far.


In [1]:
import pandas as pd
from itertools import chain
from functools import reduce

In [2]:
# Hyperparameters
POPULATION_SIZE = 150
NUM_GENERATIONS = 200
MUTATION_RATE = 0.05
EXCLUSIONS = {
    "items" : [6894,6895,3080,9031]
}
LOWER_BOUNDS = {
    # "level" : 190
}
UPPER_BOUNDS = {}

### Load

In [3]:
import json
with open('data/MAPPED_ITEMS.json', 'r') as file:
    items = {i["ankama_id"]: i for i in json.load(file)}

with open('data/MAPPED_SETS.json', 'r') as file:
    item_sets = {i["ankama_id"]: i for i in json.load(file)}


### Filters

In [4]:
items = {k:v for k,v in items.items() if not k in EXCLUSIONS["items"]}

In [5]:
pool_types = [
    (1, 1),
    (2, 1),
    (3, 2), # Ring
    (4, 1),
    (5, 1),
    (7, 1),
    (10,1),
    (11, 1),
    (12, 1),
    (13, 6),
]
pools = []
for t,count in pool_types:
    for i in range(count):
        pools.append([id for id in items if items[id]["type"]["superTypeId"] == t])
NUM_TYPES = len(pools)
print(f"Total pools: {len(pools)} with sizes {[len(pool) for pool in pools]}")

Total pools: 16 with sizes [322, 747, 369, 369, 354, 364, 178, 366, 301, 145, 327, 327, 327, 327, 327, 327]


## Effect logic dictionaries

In [9]:
def get_item_contribution(item):
    # Get the effects of the item
    item_effects = item["effects"]
    if not item_effects:
        return {}
    contributions = {}
    for d in item_effects:
        field = "max" if d["min_max_irrelevant"] == 0 else "min"
        contributions[d["element_id"]] = d[field]
    return contributions


In [10]:
def get_item_set(item):
    if not item["hasParentSet"]:
        return None
    return item_sets[item["parentSet"]["id"]]

In [11]:
def get_set_contribution(item_set,overlap):
    set_effects = item_set["effects"][str(overlap)]
    contributions = {}
    for d in set_effects:
        field = "max" if d["min_max_irrelevant"] == 0 else "min"
        contributions[d["element_id"]] = d[field]
    return contributions

## Damage logic

In [12]:
# Interpreters/Mappers to characteristics per element
damage_mapper = {
    # Damage
        189 : 36, # Air,
        214 : 22, # Water
        198 : 13, # Fire
        194 : 45, # Earth
        195 : 45, # Neutral (by earth)
    # Steal
        224 : 36, # Air,
        203 : 22, # Water
        193 : 13, # Fire
        221 : 45, # Earth
        223 : 45, # Neutral
    }

bonus_damage_mapper = {
    # Damage
        189 : 47, # Air,
        214 : 27, # Water
        198 : 61, # Fire
        194 : 48, # Earth
        195 : 49, # Neutral
    # Steal
        224 : 47, # Air,
        203 : 27, # Water
        193 : 61, # Fire
        221 : 48, # Earth
        223 : 49, # Neutral
}

In [13]:
def get_weapon_damage(weapon):
    if not weapon["effects"]:
        return {}
    damage_effects = [d for d in weapon["effects"] if d["active"]]
    if not damage_effects:
        return {}
    contributions = {}
    for d in damage_effects:
        field = "max" if d["min_max_irrelevant"] == 0 else "min"
        contributions[d["element_id"]] = d[field]
    return contributions


## 0. Methods

In [14]:
import random
def random_chromosome():
    return [random.randint(0, len(slot_pool)-1) for slot_pool in pools]

In [15]:
# Mutation
def mutate(chromosome, mutation_rate=MUTATION_RATE):
    # Create a new chromosome by mutating some slots
    new_chromosome = chromosome[:]
    for i, slot_items in enumerate(pools):
        if random.random() < mutation_rate:
            new_chromosome[i] = random.randint(0, len(slot_items)-1)
    return new_chromosome

In [16]:
# Crossover
def crossover(parent1, parent2):
    # Perform single-point crossover
    point = random.randint(1, NUM_TYPES-2)
    return parent1[:point] + parent2[point:]

In [17]:
def get_chromosome_weapon(chromosome):
    return next(items[pools[idx][i]] for idx,i in enumerate(chromosome) if items[pools[idx][i]]["type"]["superTypeId"] == 2)

In [17]:
def get_chromosome_item_contributions(chromosome):
    # Item contributions
    chromosome_items = [items[pools[idx][i]] for idx,i in enumerate(chromosome)]
    lst = [get_item_contribution(item) for item in chromosome_items]
    return pd.DataFrame(lst).sum(axis=0).to_dict()

def get_chromosome_set_contributions(chromosome):
    chromosome_items = [items[pools[idx][i]] for idx,i in enumerate(chromosome)]
    # Set effects contributions
    lst = []
    seen_sets = set()
    for item in chromosome_items:
        item_set = get_item_set(item)
        if item_set is None:
            continue
        if item_set["ankama_id"] in seen_sets:
            continue
        seen_sets.add(item_set["ankama_id"])
        overlap = len(set([i["ankama_id"] for i in chromosome_items]) & set(item_set["items"]))
        lst.append(get_set_contribution(item_set, overlap))
    return pd.DataFrame(lst).sum(axis=0).to_dict()

In [18]:
# Fitness function
def get_totals(chromosome):
    # Get the item contributions
    item_contributions = get_chromosome_item_contributions(chromosome)

    # Get the set contributions
    set_contributions = get_chromosome_set_contributions(chromosome)
    return pd.DataFrame([item_contributions, set_contributions]).sum(axis=0).to_dict()

In [19]:
def get_damage(weapon, chromosome):
    totals = get_totals(chromosome)

    # Character stats
    power = totals.get("32",0)
    base_crit_chance = totals.get("29",0)
    base_crit_added_damage = totals.get("38",0)

    # Critical hit logic
    crit_bonus = weapon["criticalHitBonus"] if not pd.isna(weapon["criticalHitBonus"]) else 0
    weapon_crit_chance = weapon["criticalHitProbability"] if not pd.isna(weapon["criticalHitProbability"]) else 0
    crit_chance = max(min(weapon_crit_chance + base_crit_chance,100), 0)  # Ensure crit_chance is between 0 and 100
    expected_crit_bonus = max(crit_bonus * crit_chance/100, 0) # Expected value of distributionß
    expected_crit_added_damage = max(base_crit_added_damage * crit_chance/100, 0) # Expected value of distribution
    # max_crit_bonus = crit_bonus # Use this for maximum possible

    weapon_damage = get_weapon_damage(weapon)
    # Calculate total damage
    damage = 0
    for element_id,weapon_base_damage in weapon_damage.items():
        # Get the stat and base damage
        if element_id not in damage_mapper:
            continue
        stat = damage_mapper[element_id]
        stat_base = totals.get(stat,0)

        # Get the added bonus damage from the stat
        bonus_damage_id = bonus_damage_mapper[element_id]
        added_bonus = totals.get(bonus_damage_id, 0)
        damage += (weapon_base_damage + expected_crit_bonus) * (1+(stat_base+power)/100) + expected_crit_added_damage + added_bonus
    return damage
    

In [20]:
def fitness(chromosome):
    # Penalizations
    if len(set(chromosome[-pool_types[-1][1]:])) < pool_types[-1][1]:
        # Some dofus or trophies are duplicated
        return 0
    if len(set(chromosome[2:4])) < 2:
        # Some rings are duplicated
        return 0
    # TODO: Penalize two rings
    # return get_totals(chromosome).get(18, 0)
    weapon = get_chromosome_weapon(chromosome)
    return get_damage(weapon,chromosome)

In [21]:
# Sanity check to make sure the function works on random chromosomes
for _ in range(10):
    chromosome = random_chromosome()
    assert isinstance(fitness(chromosome), float) or isinstance(fitness(chromosome), int)

## 1. Initialize population

In [22]:
population = [random_chromosome() for _ in range(POPULATION_SIZE)]

## 2. Selection

In [23]:
def select(population, k=10):
    # Select top k chromosomes
    scored = sorted(population, key=fitness, reverse=True)
    return scored[:k]

## 3. Main Loop

In [24]:
from tqdm import tqdm
for gen in tqdm(range(NUM_GENERATIONS), desc="Generations"):
    # Select best individuals
    selected = select(population, k=10)
    # Generate new population
    new_population = selected[:]
    while len(new_population) < POPULATION_SIZE:
        parent1, parent2 = random.sample(selected, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    population = new_population
    # Print best fitness in this generation
    best = max(population, key=fitness)
    # print(f'Gen {gen}: Best damage = {fitness(best)}')

Generations: 100%|██████████| 200/200 [00:49<00:00,  4.02it/s]


In [ ]:
fitness(best)

1479.775

In [25]:
best_items = [items[pools[idx][i]] for idx,i in enumerate(best)]
[(i["type"]["name"]["es"],i["name"]["es"])for i in best_items]

[('Amuleto', 'Collar de tritáceo'),
 ('Espada', 'Nueve Colas'),
 ('Anillo', 'Anillo del conde Kontatrás'),
 ('Anillo', 'Anillo crústico'),
 ('Cinturón', 'Bosque del lich'),
 ('Bota', 'Botas del conde Kontatrás'),
 ('Escudo', 'Paravela de Mami Ayuto'),
 ('Sombrero', 'Pañoleta de Mami Ayuto'),
 ('Capa', 'Capa Pología'),
 ('Mascotura', 'Kompost'),
 ('Trofeo', 'Hércules'),
 ('Trofeo', 'Rabioso mayor'),
 ('Trofeo', 'Hércules mayor'),
 ('Dofus', 'Dolmanax'),
 ('Trofeo', 'Rabioso'),
 ('Trofeo', 'Furibundo mayor')]

In [26]:
lst = [get_item_contribution(i) for i in best_items]
df = pd.DataFrame(lst,index=[i["name"]["es"] for i in best_items])

In [27]:
# Count the sets
set_counts = pd.Series([i["parentSet"]["name"]["es"] for i in best_items if i["hasParentSet"]]).value_counts()
set_counts

Set de la sima                    2
Set del conde Kontatrás           2
Set de Mami Ayuto                 2
Reliquias de la Aurora Púrpura    1
Name: count, dtype: int64

In [28]:
item_descriptions = {}
for _,item in items.items():
    if not item["effects"]:
        continue
    for effect in item["effects"]:
        item_descriptions[effect["element_id"]] = effect["type"]["es"]

In [29]:
for k,v in get_totals(best).items():
    print(f"({k}) {item_descriptions[k]}: {v}")

(9) vitalidad: 3099.0
(45) fuerza: 1449.0
(10) sabiduría: 305.0
(25) prospección: 15.0
(48) de daño de tierra: 120.0
(49) de daño neutro: 100.0
(16) % resistencia al aire: 43.0
(26) placaje: 76.0
(12) PA: 1.0
(195) de daños neutrales: 99.0
(36) agilidad: 144.0
(37) % resistencia al fuego: 19.0
(38) de daño crítico: 9.0
(31) alcance: -1.0
(29) % crítico: 20.0
(24) iniciativa: -401.0
(46) de resistencia a críticos: 49.0
(17) % resistencia al agua: 24.0
(64) al retiro de PA: 20.0
(13) inteligencia: 160.0
(22) suerte: 115.0
(70) de resistencia al empuje: 68.0
(59) huida: 27.0
(61) de daño de fuego: 20.0
(27) de daño de agua: 10.0
(47) de daño de aire: 10.0
(63) % resistencia a la tierra: 25.0
(39) a la esquiva pérdidas de PM: -13.0
(75) a la esquiva pérdidas de PA: 3.0
(8) PM: 3.0
(33) de resistencia al neutro: 30.0
(121) de cura: 55.0
(62) de daño de empuje: 85.0
(34) % resistencia neutral: 20.0
(98) Actitud: 109.0
(0) Intercambiable:: 0.0


In [30]:
assert False

AssertionError: 

In [ ]:
weapon = get_chromosome_weapon(best)
get_damage(weapon,best)

np.float64(2228.7999999999997)

In [ ]:
best

[482, 320, 350, 310, 297, 184, 89, 119, 264, 226, 5, 265, 95, 107, 2, 0]

In [ ]:
item = tmp.iloc[8]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Anillo de Mizz Frizz


,name,effectId,diceNum,diceSide
2861093746004760000,#1{{~1~2 a }}#2 vitalidad,125,251.0,300.0
2861093746004760001,#1{{~1~2 a }}#2 fuerza,118,31.0,50.0
2861093746004760002,#1{{~1~2 a }}#2 inteligencia,126,31.0,50.0
2861093746004760003,#1{{~1~2 a }}#2 suerte,123,31.0,50.0
2861093746004760004,#1{{~1~2 a }}#2 % crítico,115,2.0,3.0
2861093746004760005,#1{{~1~2 a }}#2 de cura{{~p}}{{~z}},178,7.0,10.0
2861093746004760006,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} neutro,430,7.0,10.0
2861093746004760007,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} de tierra,422,7.0,10.0
2861093746004760008,#1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} a l...,240,7.0,10.0
2861093746004760009,-#1{{~1~2 a -}}#2 huida,754,7.0,10.0


In [ ]:
item = tmp.iloc[0]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Espada maldita del gran Desangrador Guerrero


,name,effectId,diceNum,diceSide
2861093746004750990,#1{{~1~2 a }}#2 de daños de aire,98,9.0,18.0
2861093746004750991,#1{{~1~2 a }}#2 de daños de agua,96,9.0,18.0
2861093746004750992,#1{{~1~2 a }}#2 de daños de fuego,99,9.0,18.0
2861093746004750993,#1{{~1~2 a }}#2 de daños neutrales,100,9.0,18.0
2861093746004750994,#1{{~1~2 a }}#2 de daños de tierra,97,9.0,18.0
2861093746004750995,#1{{~1~2 a }}#2 suerte,123,30.0,0.0
2861093746004750996,#1{{~1~2 a }}#2 inteligencia,126,30.0,0.0
2861093746004750997,#1{{~1~2 a }}#2 fuerza,118,30.0,0.0
2861093746004750998,#1{{~1~2 a }}#2 agilidad,119,30.0,0.0
2861093746004750999,#1{{~1~2 a }}#2 sabiduría,124,30.0,0.0


In [ ]:
effects.loc["98"]

name                        #1{{~1~2 a }}#2 de daños de aire
type                                                 Effects
descriptionId                                        1102898
iconId                                                     6
characteristic                                             0
category                                                   2
characteristicOperator                                   NaN
showInTooltip                                              1
useDice                                                    1
forceMinMax                                                1
boost                                                      0
active                                                     0
oppositeId                                                 0
theoreticalDescriptionId                                 NaN
theoreticalPattern                                         1
showInSet                                                  0
bonusType               

In [ ]:
contributions[item.name]

characteristic
0     -90.0
10     30.0
11    165.0
12     30.0
13     30.0
14     30.0
15     30.0
33      5.0
34      5.0
35      5.0
36      5.0
37      5.0
Name: effectValue, dtype: float64

In [ ]:
item = tmp.iloc[8]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Anillo de Mizz Frizz


,name,effectId,diceNum,diceSide
2861093746004760000,#1{{~1~2 a }}#2 vitalidad,125,251.0,300.0
2861093746004760001,#1{{~1~2 a }}#2 fuerza,118,31.0,50.0
2861093746004760002,#1{{~1~2 a }}#2 inteligencia,126,31.0,50.0
2861093746004760003,#1{{~1~2 a }}#2 suerte,123,31.0,50.0
2861093746004760004,#1{{~1~2 a }}#2 % crítico,115,2.0,3.0
2861093746004760005,#1{{~1~2 a }}#2 de cura{{~p}}{{~z}},178,7.0,10.0
2861093746004760006,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} neutro,430,7.0,10.0
2861093746004760007,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} de tierra,422,7.0,10.0
2861093746004760008,#1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} a l...,240,7.0,10.0
2861093746004760009,-#1{{~1~2 a -}}#2 huida,754,7.0,10.0


In [ ]:
contributions[item.name]

characteristic
10     50.0
11    300.0
13     50.0
15     50.0
18      3.0
49     10.0
54     10.0
78    -10.0
86     10.0
88     10.0
92     10.0
Name: effectValue, dtype: float64

In [ ]:
best

[482, 320, 350, 310, 297, 184, 89, 119, 264, 226, 5, 265, 95, 107, 2, 0]

In [ ]:
lst = get_totals(best)
for idx,i in enumerate(lst):
    item = tmp.iloc[idx]
    print(f"{item['name']} (rid:{item.name}):")
    print(i.get("33", 0))  # Assuming "54" is a characteristic ID you want to print

Espada maldita del gran Desangrador Guerrero (rid:2861093746004724544):
5.0
Collar rhoarim (rid:2861093746004741472):
0
Taparrabos rhoarim (rid:2861093746004741473):
0
Botas de Miauvizor (rid:2861093746004731252):
0
Casco de Miauvizor (rid:2861093746004731253):
10.0
Capa Brigada (rid:2861093746004727614):
0
Lokulto (rid:2861093746004728051):
0
Cuadrifolio (rid:2861093746004732259):
0
Anillo de Mizz Frizz (rid:2861093746004728415):
0
Anillo Gwen Astarde (rid:2861093746004726812):
0
Dofus de los hielos (rid:2861093746004723117):
0
Lozano mayor (rid:2861093746004730368):
0
Dolmanax (rid:2861093746004727753):
0
Hércules mayor (rid:2861093746004728123):
0
Dofus Turquesa (rid:2861093746001847037):
0
Dofus Púrpura (rid:2861093746001847005):
0


In [ ]:
contributions["2861093746004726812"]

characteristic
10     60.0
12     30.0
13     60.0
18      5.0
26      2.0
44    400.0
48     10.0
82     10.0
84     15.0
88     15.0
90     15.0
92     15.0
Name: effectValue, dtype: float64

In [ ]:
item = items.loc["2861093746004730368"]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]
# get_item_contribution(item)

Lozano mayor


,name,effectId,diceNum,diceSide
2861093746004764621,#1{{~1~2 a }}#2 potencia (daños generales),138,80.0,0.0


In [ ]:
for i in items["criteria"].unique():
    print(i)

nan
CM<5
CP<12&CM<6&CW>99
CP<11
CP<12
CP<8
cs<25
ci<25
PT!24006&PT!350
PT!368&PT!24029
PT!369&PT!24017
PT!370&PT!24027
Ps=1&Pa>20
CP<9
Ps=2&Pa>30
cs<50
cs<80
cs<100
ca<25
ca<50
ca<80
ca<100
cw<25
cw<50
cw<80
cw<100
cv<25
cv<50
cv<80
cc<25
cv<100
cc<50
cc<80
cc<100
ci<50
ci<80
ci<100
CP<10
Pz=0&Sc!336
CM<6
Ps=1&Pa>10
Pj=44|Pj=48
CC>10
PG=3
CS>99&CA>99&CV>99
PE!8
PJ>2,40|PJ>24,40|PJ>36,40|PJ>28,40|PJ>26,40|PJ>41,40
PJ>2,80|PJ>24,80|PJ>36,80|PJ>28,80|PJ>26,80|PJ>41,80
PJ>2,120|PJ>24,120|PJ>36,120|PJ>28,120|PJ>26,120|PJ>41,120
PJ>2,180|PJ>24,180|PJ>36,180|PJ>28,180|PJ>26,180|PJ>41,180
PZ=1
CI<100&CC<100
BI=1
ci<20
ci<10
cw<20
cw<30
cw<60
ci<30
ci<5
cw<5
ci<60
cw<10
cw<40
ci<70
ci<40
cv<5
cv<30
cv<40
cv<10
cv<20
Ps=2
Ps=1
PN~Silaisie&BI=1
Ps=0
PO!2181&PO!2177&Ps=1&Pa=27&BI=1
PO!2177
PO=2177
PO=2177&Ps=1&Pa=27
CS<40&CI>80
PO!1723
CW>49&Ps=1
CW>49&Ps=2
CA>50
CC>50
CI>90
CS>90
CW>30&CV>50
Po!45
CC>30
CI>299&CC>199
CI>199
PS=0
PS=1
PG=1
PG=2
PG=4
PG=5
PG=6
PG=7
PG=8
PG=9
PG=10
PG=11
PN~Lordsoth

In [ ]:
item

name           Lozano mayor
type                  Items
m_flags               24712
id                    16332
nameId               515826
                   ...     
diceNum                 NaN
diceSide                NaN
displayZero             NaN
class                Trofeo
set                     NaN
Name: 2861093746004730368, Length: 69, dtype: object